#### 1928. Minimum Cost to Reach Destination in Time

* https://leetcode.com/problems/minimum-cost-to-reach-destination-in-time/description/


In [ ]:
class Solution:
    """
        Algorithm
            Build graph
            Min-heap: (cost, time, node)
            Track:
            min_time[node] = smallest time we’ve reached this node
            For each neighbor:
            new_time = time + edge_time
            new_cost = cost + fee
            Push if:
            new_time <= maxTime
            AND improves over known state
        
        Mental Model -
            Problem Type	                    Strategy
            shortest path (min sum)	            Dijkstra
            constraints (time, stops, fuel)	    state augmentation
            multi-dimension optimization	    heap + pruning

    """
    def minCost(self, maxTime: int, edges: List[List[int]], passingFees: List[int]) -> int:
        """
        Dijkstra on augmented state: (cost, time, node)

        - cost is minimized (primary objective)
        - time is constrained (secondary constraint)

        We prune dominated states using min_time array.

        TC - O(E * maxTime * log (N * maxTime))
        SC - O(N * maxTime + E)
        """
        n = len(passingFees)
        
        # Step 1: Build adjacency list
        graph = defaultdict(list)
        for src, dest, time in edges:
            graph[src].append((dest, time))
            graph[dest].append((src, time))
        

        # Step 2: Min heap -> (total_cost, current_time, node)
        minheap = [(passingFees[0], 0, 0)] # cost, time, src node

        # Step 3: Track best time to reach node
        # Used for pruning worse states
        # Always remember - Dijkstra's == Heap + Best (list/visitedset/2D matrix)
        best_min_time = [float('inf')]*n

        while minheap:
            curr_cost, curr_time, node = heapq.heappop(minheap)

            if node == n-1:
                return curr_cost

            for nei, time in graph[node]:
                new_time = time + curr_time

                if new_time <= maxTime and new_time < best_min_time[nei]:
                    new_cost = curr_cost + passingFees[nei]
                    best_min_time[nei] = new_time
                    heapq.heappush(minheap, (new_cost, new_time, nei))
    
        return -1

🧠 Your Thought

“Why not track min_cost[node] and use that to prune, as long as time ≤ maxTime?”

Sounds reasonable… but it’s incorrect.

❌ Why min_cost[node] is NOT sufficient

Because:

A cheaper path to a node may consume too much time, making it useless later.

🔥 Counterexample (this is what interviewers expect)
maxTime = 30

Graph:
0 → 1 → 3
 \      ↑
  → 2 →
Paths:
Path A (cheap but slow)
0 → 1 → 3
cost = 5
time = 29
Path B (slightly expensive but fast)
0 → 2 → 3
cost = 8
time = 10
Now the trap:

If you store:

min_cost[3] = 5   # from Path A

You will discard Path B because:

8 > 5  ❌

BUT…

👉 Path A leaves you almost no time for future edges
👉 Path B is actually the only viable path in many cases

💥 Core Issue

Cost alone doesn’t define a “good state”.

You must consider:

(node, time)  ← NOT just node
✅ Correct Dominance Rule

A state (cost1, time1) is better than (cost2, time2) only if:

cost1 <= cost2 AND time1 <= time2

👉 This is multi-dimensional optimization

⚖️ Why we track min_time[node] instead

We flip thinking:

Instead of:
minimize cost at node

We use:

minimize time at node (for pruning)
while heap ensures cost optimality
✅ Why min_time[] works

When we do:

if new_time >= min_time[node]:
    skip

We’re saying:

“We already reached this node faster, so this slower path is useless.”

✔ Faster arrival = more flexibility later
✔ Keeps future options open

🧠 Key Insight (THIS is the takeaway)
Metric	Role
cost	optimization target (handled by heap)
time	constraint → used for pruning
🚨 Why not track both?

You can:

Full DP (always correct)
dp[node][time] = min_cost

But:

Time complexity = O(N * maxTime)

Often too slow

⚡ Why heap + min_time is enough here

Because:

We always expand lowest cost first (Dijkstra property)

We prune only when strictly worse in time

👉 This avoids missing optimal solutions

🧩 Mental Model Upgrade

When you see:

❓ “Why not track X?”

Ask:

“Can two states with same node but different X lead to different futures?”

If YES → you need multi-state tracking

🆚 Compare with your previous confusion (important)
Problem	Why 1D works
LC 1514 (max prob)	probability is monotonic
LC 1631 (min effort)	effort is path-max → once fixed, final
LC 1928	❌ tradeoff → cost vs time
🏁 Final Answer (interview-ready)

We cannot use min_cost[node] because a cheaper path may consume too much time, making it invalid for reaching the destination within constraints.

Instead, we treat time as a constraint and track the minimum time to reach each node, while using a min-heap to ensure we always explore the lowest-cost paths first.

This avoids pruning potentially optimal solutions that require slightly higher cost but significantly less time.